# EduVision_DV — 04. KPI Engineering

## Part A — Build fact tables

In [4]:
import pandas as pd
import os

CLEANED_DIR = os.path.join(".", "..", "data", "cleaned")


def main():
    dim_u = pd.read_csv(os.path.join(CLEANED_DIR, "dim_university.csv"))
    qs = pd.read_csv(os.path.join(CLEANED_DIR, "qs_2025_clean.csv")).set_index("institution_name")
    the = pd.read_csv(os.path.join(CLEANED_DIR, "the_2024_clean.csv")).set_index("name")
    wur = pd.read_csv(os.path.join(CLEANED_DIR, "wur_2023_clean.csv")).set_index("name_of_university")

    # ------------------------------------------------------------------
    # fact_university_performance: rank + overall score + reputation
    # ------------------------------------------------------------------
    print("=" * 60, "\nfact_university_performance\n" + "=" * 60)
    perf_rows = []
    for _, u in dim_u.iterrows():
        if pd.notna(u["qs_name"]) and u["qs_name"] in qs.index:
            q = qs.loc[u["qs_name"]]
            perf_rows.append({
                "university_id": u["university_id"], "year": 2025, "source": "QS_2025",
                "global_rank": q["rank_numeric"], "overall_score": q["overall_score"],
                "academic_reputation": q["academic_reputation_score"],
                "employer_reputation": q["employer_reputation_score"],
            })
        if pd.notna(u["the_name"]) and u["the_name"] in the.index:
            t = the.loc[u["the_name"]]
            perf_rows.append({
                "university_id": u["university_id"], "year": 2024, "source": "THE_2024",
                "global_rank": t["rank_numeric"], "overall_score": t["scores_overall"],
                "academic_reputation": pd.NA,   # THE doesn't publish this field - left blank, not invented
                "employer_reputation": pd.NA,
            })
        if pd.notna(u["wur_name"]) and u["wur_name"] in wur.index:
            w = wur.loc[u["wur_name"]]
            perf_rows.append({
                "university_id": u["university_id"], "year": 2023, "source": "WUR_2023",
                "global_rank": w["rank_numeric"], "overall_score": w["overall_score"],
                "academic_reputation": pd.NA,
                "employer_reputation": pd.NA,
            })
    fact_performance = pd.DataFrame(perf_rows)
    fact_performance.to_csv(os.path.join(CLEANED_DIR, "fact_university_performance.csv"), index=False)
    print(f"rows: {len(fact_performance)} | saved -> fact_university_performance.csv")

    # ------------------------------------------------------------------
    # fact_research: research + citation indicators only
    # ------------------------------------------------------------------
    print("\n" + "=" * 60, "\nfact_research\n" + "=" * 60)
    research_rows = []
    for _, u in dim_u.iterrows():
        if pd.notna(u["qs_name"]) and u["qs_name"] in qs.index:
            q = qs.loc[u["qs_name"]]
            research_rows.append({
                "university_id": u["university_id"], "year": 2025, "source": "QS_2025",
                "research_score": pd.NA,  # QS has no direct "research score" - citations per faculty is the closest documented proxy
                "citation_score": q["citations_per_faculty_score"],
            })
        if pd.notna(u["the_name"]) and u["the_name"] in the.index:
            t = the.loc[u["the_name"]]
            research_rows.append({
                "university_id": u["university_id"], "year": 2024, "source": "THE_2024",
                "research_score": t["scores_research"],
                "citation_score": t["scores_citations"],
            })
        if pd.notna(u["wur_name"]) and u["wur_name"] in wur.index:
            w = wur.loc[u["wur_name"]]
            research_rows.append({
                "university_id": u["university_id"], "year": 2023, "source": "WUR_2023",
                "research_score": w["research_score"],
                "citation_score": w["citations_score"],
            })
    fact_research = pd.DataFrame(research_rows)

    # research_impact: documented, transparent composite (NOT Sustainability Score
    # per the doc's explicit warning). Only computed where both inputs exist.
    fact_research["research_score"] = pd.to_numeric(fact_research["research_score"], errors="coerce")
    fact_research["citation_score"] = pd.to_numeric(fact_research["citation_score"], errors="coerce")
    fact_research["research_impact"] = (
        fact_research[["research_score", "citation_score"]]
        .mean(axis=1, skipna=False)
    )


    fact_research.to_csv(os.path.join(CLEANED_DIR, "fact_research.csv"), index=False)
    print(f"rows: {len(fact_research)} | saved -> fact_research.csv")
    print(f"research_impact populated: {fact_research['research_impact'].notna().sum()} / {len(fact_research)}")

    # ------------------------------------------------------------------
    # fact_student: student counts, ratios, international %
    # ------------------------------------------------------------------
    print("\n" + "=" * 60, "\nfact_student\n" + "=" * 60)
    student_rows = []
    for _, u in dim_u.iterrows():
        if pd.notna(u["the_name"]) and u["the_name"] in the.index:
            t = the.loc[u["the_name"]]
            student_rows.append({
                "university_id": u["university_id"], "year": 2024, "source": "THE_2024",
                "total_students": t["stats_number_students"],
                "students_per_staff": t["stats_student_staff_ratio"],
                # THE gives a % directly - labeled as a true percentage, not a score
                "international_student_percentage": t["stats_pc_intl_students"],
                "international_student_is_true_percentage": True,
            })
        if pd.notna(u["wur_name"]) and u["wur_name"] in wur.index:
            w = wur.loc[u["wur_name"]]
            student_rows.append({
                "university_id": u["university_id"], "year": 2023, "source": "WUR_2023",
                "total_students": w["no_of_student"],
                "students_per_staff": w["no_of_student_per_staff"],
                "international_student_percentage": w["international_student"],
                "international_student_is_true_percentage": True,
            })
        if pd.notna(u["qs_name"]) and u["qs_name"] in qs.index:
            q = qs.loc[u["qs_name"]]
            student_rows.append({
                "university_id": u["university_id"], "year": 2025, "source": "QS_2025",
                "total_students": pd.NA,
                "students_per_staff": pd.NA,  # QS gives a *score*, not the raw ratio - see note below
                # QS publishes a 0-100 SCORE, not a percentage. Kept separate and
                # flagged so it never gets treated as a true percentage in Tableau.
                "international_student_percentage": pd.NA,
                "international_student_is_true_percentage": False,
                "international_student_score_qs": q["international_students_score"],
                "faculty_student_score_qs": q["faculty_student_score"],
            })
    fact_student = pd.DataFrame(student_rows)
    fact_student.to_csv(os.path.join(CLEANED_DIR, "fact_student.csv"), index=False)
    print(f"rows: {len(fact_student)} | saved -> fact_student.csv")

    # ------------------------------------------------------------------
    # fact_country_education: World Bank indicators, pivoted wide by indicator,
    # keyed by country_id (built in Step 3 - only if the file was provided)
    # ------------------------------------------------------------------
    print("\n" + "=" * 60, "\nfact_country_education\n" + "=" * 60)
    wb_matched_path = os.path.join(CLEANED_DIR, "world_bank_matched.csv")
    if os.path.exists(wb_matched_path):
        wb = pd.read_csv(wb_matched_path)
        wb = wb.dropna(subset=["country_id"])  # unmatched countries stay logged, not guessed into the model
        fact_country_education = wb.pivot_table(
            index=["country_id", "country_clean"], columns="indicator_name", values="value"
        ).reset_index().rename(columns={"country_clean": "country_name"})
        fact_country_education.to_csv(os.path.join(CLEANED_DIR, "fact_country_education.csv"), index=False)
        print(f"rows: {len(fact_country_education)} | countries: {fact_country_education['country_id'].nunique()} "
              f"| saved -> fact_country_education.csv")
    else:
        print("world_bank_matched.csv not found yet - fact_country_education.csv will be created "
              "once the World Bank file is added and Steps 2-3 are re-run. Nothing else depends on it.")


if __name__ == "__main__":
    main()


fact_university_performance
rows: 6409 | saved -> fact_university_performance.csv

fact_research
rows: 6409 | saved -> fact_research.csv
research_impact populated: 3625 / 6409

fact_student
rows: 6409 | saved -> fact_student.csv

fact_country_education
rows: 119 | countries: 119 | saved -> fact_country_education.csv


## Part B — Engineer the KPIs

In [5]:
import pandas as pd
import os

CLEANED_DIR = os.path.join(".", "..", "data", "cleaned")
FINAL_DIR = os.path.join(".", "..", "data", "final")
DOCS_DIR = os.path.join(".", "..", "docs")
os.makedirs(FINAL_DIR, exist_ok=True)


def main():
    dim_u = pd.read_csv(os.path.join(CLEANED_DIR, "dim_university.csv"))
    perf = pd.read_csv(os.path.join(CLEANED_DIR, "fact_university_performance.csv"))
    research = pd.read_csv(os.path.join(CLEANED_DIR, "fact_research.csv"))
    student = pd.read_csv(os.path.join(CLEANED_DIR, "fact_student.csv"))

    print("=" * 60, "\nKPI 1: GLOBAL RANKING SCORE\n" + "=" * 60)
    # Source: overall_score of whichever ranking source the row came from.
    # Rescaled 0-100 WITHIN each source+year, since QS and THE overall
    # scores aren't on directly comparable scales to begin with.
    perf["global_ranking_score"] = perf.groupby(["source", "year"])["overall_score"].transform(
        lambda s: (s - s.min()) / (s.max() - s.min()) * 100 if s.notna().any() and s.max() != s.min() else pd.NA
    ).round(2)
    print(f"Populated: {perf['global_ranking_score'].notna().sum()} / {len(perf)}")
    print("Missing-value treatment: left NaN when overall_score itself is missing "
          "(both QS and THE only publish an exact score for their top ranking tier) - never filled with 0.")

    print("\n" + "=" * 60, "\nKPI 2 & 6: RESEARCH IMPACT SCORE / RESEARCH PRODUCTIVITY INDEX\n" + "=" * 60)
    # research_impact already built in fact_research.csv as avg(research_score, citation_score)
    research["research_productivity_index"] = (
        research.groupby(["source", "year"])["research_impact"]
        .transform(lambda s: s.rank(pct=True) * 100)
        .round(1)
    )
    print(f"research_impact_score populated: {research['research_impact'].notna().sum()} / {len(research)}")
    print(f"research_productivity_index populated: {research['research_productivity_index'].notna().sum()} / {len(research)}")
    print("Formula (documented, transparent, reproducible): "
          "research_impact = mean(research_score, citation_score); "
          "research_productivity_index = percentile rank of research_impact within (source, year). "
          "Deliberately NOT Sustainability_Score, per project reference guide Section 22.")

    print("\n" + "=" * 60, "\nKPI 3: FACULTY-TO-STUDENT RATIO\n" + "=" * 60)
    # Actual ratio only (students_per_staff) - QS's Faculty_Student_Score is
    # kept as a separate labeled column and NEVER presented as the ratio.
    student["faculty_student_ratio"] = student["students_per_staff"]
    print(f"Populated (true ratio, THE + WUR2023 rows only): {student['faculty_student_ratio'].notna().sum()} / {len(student)}")
    print(f"QS rows instead carry 'faculty_student_score_qs' (0-100 score, NOT a ratio) - "
          f"{student['faculty_student_score_qs'].notna().sum()} populated")

    print("\n" + "=" * 60, "\nKPI 4: INTERNATIONAL STUDENT PERCENTAGE\n" + "=" * 60)
    # Only rows where the source actually reports a true percentage (THE, WUR2023)
    # get this KPI. QS's score is kept as 'international_student_score_qs'.
    print(f"True-percentage rows populated: {student['international_student_percentage'].notna().sum()} / {len(student)}")
    print(f"QS score-only rows: {student['international_student_score_qs'].notna().sum()} "
          f"(labeled 'international_student_score_qs', never renamed to look like a percentage)")

    print("\n" + "=" * 60, "\nKPI 5: ACADEMIC REPUTATION SCORE\n" + "=" * 60)
    # Direct from QS - the only one of the three sources that publishes this indicator.
    print(f"Populated (QS_2025 rows only): {perf.loc[perf['source']=='QS_2025', 'academic_reputation'].notna().sum()} / "
          f"{(perf['source']=='QS_2025').sum()}")

    # ------------------------------------------------------------------
    # Assemble the final KPI table: one row per university_id x year x source
    # ------------------------------------------------------------------
    kpi = perf.merge(research.drop(columns=["research_score", "citation_score"]),
                      on=["university_id", "year", "source"], how="outer")
    kpi = kpi.merge(student, on=["university_id", "year", "source"], how="outer")
    kpi = kpi.merge(
        dim_u[["university_id", "display_name", "country_id", "country_name", "region"]],
        on="university_id", how="left"
    )

    kpi = kpi.rename(columns={
        "overall_score": "overall_score_source_native",
        "academic_reputation": "academic_reputation_score",
        "research_impact": "research_impact_score",
    })

    final_cols = [
        "university_id", "display_name", "country_id", "country_name", "region",
        "source", "year",
        "global_rank", "global_ranking_score", "overall_score_source_native",
        "academic_reputation_score", "employer_reputation",
        "research_impact_score", "research_productivity_index",
        "faculty_student_ratio", "faculty_student_score_qs",
        "international_student_percentage", "international_student_score_qs",
        "international_student_is_true_percentage",
        "total_students",
    ]
    kpi_final = kpi[[c for c in final_cols if c in kpi.columns]]

    out_path = os.path.join(FINAL_DIR, "eduvision_final_dataset.xlsx")
    with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
        kpi_final.to_excel(writer, sheet_name="kpi_dataset", index=False)
        dim_u.to_excel(writer, sheet_name="dim_university", index=False)
        pd.read_csv(os.path.join(CLEANED_DIR, "dim_country.csv")).to_excel(writer, sheet_name="dim_country", index=False)
        perf.to_excel(writer, sheet_name="fact_university_performance", index=False)
        research.to_excel(writer, sheet_name="fact_research", index=False)
        student.to_excel(writer, sheet_name="fact_student", index=False)

        country_ed_path = os.path.join(CLEANED_DIR, "fact_country_education.csv")
        if os.path.exists(country_ed_path):
            country_ed = pd.read_csv(country_ed_path)
            country_ed.to_excel(writer, sheet_name="fact_country_education", index=False)
            print(f"fact_country_education: {len(country_ed)} countries added as its own sheet")
        else:
            print("fact_country_education not yet available - workbook built without it "
                  "(add the World Bank raw file and re-run Steps 2-5 to include it)")

    print(f"\nSaved multi-sheet workbook -> {out_path}")
    print(f"kpi_dataset rows: {len(kpi_final)}, columns: {len(kpi_final.columns)}")


if __name__ == "__main__":
    main()


KPI 1: GLOBAL RANKING SCORE
Populated: 1000 / 6409
Missing-value treatment: left NaN when overall_score itself is missing (both QS and THE only publish an exact score for their top ranking tier) - never filled with 0.

KPI 2 & 6: RESEARCH IMPACT SCORE / RESEARCH PRODUCTIVITY INDEX
research_impact_score populated: 3625 / 6409
research_productivity_index populated: 3625 / 6409
Formula (documented, transparent, reproducible): research_impact = mean(research_score, citation_score); research_productivity_index = percentile rank of research_impact within (source, year). Deliberately NOT Sustainability_Score, per project reference guide Section 22.

KPI 3: FACULTY-TO-STUDENT RATIO
Populated (true ratio, THE + WUR2023 rows only): 4881 / 6409
QS rows instead carry 'faculty_student_score_qs' (0-100 score, NOT a ratio) - 1503 populated

KPI 4: INTERNATIONAL STUDENT PERCENTAGE
True-percentage rows populated: 4876 / 6409
QS score-only rows: 1445 (labeled 'international_student_score_qs', never rena